# P10.6-AI — Notebook 58: split interno foraminal

Construye un split 70/15/15 por `study_id` para estrechamiento foraminal izquierdo/derecho en Sagittal T1. Audita primero el split común del Notebook 54 y, si falla en estratos raros, selecciona un split específico mediante búsqueda multilabel reproducible.

No entrena, no descarga DICOM y no accede al test oficial.

## Recursos

- Colab estándar con **CPU**.
- Autorizar Google Drive.
- No requiere GPU, token de Kaggle ni token de GitHub.
- Requiere los outputs aprobados del Notebook 57.

In [1]:
# 1) Montar Drive y actualizar la rama
from __future__ import annotations
import json, subprocess, sys
from pathlib import Path
from google.colab import drive  # type: ignore

drive.mount("/content/drive", force_remount=False)
REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
if not (REPO_ROOT / ".git").exists():
    subprocess.check_call(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_ROOT)])
else:
    subprocess.check_call(["git", "fetch", "origin", REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git", "checkout", REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_ROOT)
REPO_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
sys.path.insert(0, str(REPO_ROOT / "ai_service"))
print({"repoRef": REPO_REF, "repoSha": REPO_SHA, "gpuRequired": False})

Mounted at /content/drive
{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': 'f4cdba8c844c296ebf28fce414a6ad82394c6677', 'gpuRequired': False}


In [2]:
# 2) Resolver artefactos de entrada y salida
from pfi_ai_service.training.rsna_foraminal_split import run_split

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
RESULTS_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
SOURCE_ROOT = RESULTS_ROOT / "notebook57_foraminal_preflight"
SOURCE_MANIFEST = SOURCE_ROOT / "foraminal_candidate_manifest.csv"
SOURCE_SUMMARY = SOURCE_ROOT / "foraminal_preflight_summary.json"
COMMON_SPLIT = RESULTS_ROOT / "notebook54_split" / "study_splits.csv"
OUTPUT_ROOT = RESULTS_ROOT / "notebook58_foraminal_split"
print({"sourceManifest": str(SOURCE_MANIFEST), "commonSplit": str(COMMON_SPLIT), "outputRoot": str(OUTPUT_ROOT)})

{'sourceManifest': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook57_foraminal_preflight/foraminal_candidate_manifest.csv', 'commonSplit': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook54_split/study_splits.csv', 'outputRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/notebook58_foraminal_split'}


In [3]:
# 3) Ejecutar auditoría y búsqueda reproducible del split
result = run_split(
    SOURCE_MANIFEST,
    SOURCE_SUMMARY,
    COMMON_SPLIT,
    OUTPUT_ROOT,
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
    seed=2026,
    candidate_count=2000,
)
print(json.dumps({
    "status": "APPROVED_FOR_NOTEBOOK_59" if result["approved"] else "SPLIT_REVIEW_REQUIRED",
    "approved": result["approved"],
    "nextNotebook": result["nextNotebook"],
    "splitPolicy": result["splitPolicy"],
    "eligibleStudies": result["eligibleStudies"],
    "eligibleRows": result["eligibleRows"],
    "splits": result["splits"],
    "candidateSearch": result["candidateSearch"],
    "gateResults": result["gateResults"],
}, indent=2, ensure_ascii=False))

{
  "status": "APPROVED_FOR_NOTEBOOK_59",
  "approved": true,
  "nextNotebook": 59,
  "splitPolicy": "task_specific_multilabel_search",
  "eligibleStudies": 1972,
  "eligibleRows": 19689,
  "splits": {
    "train": {
      "studies": 1380,
      "rows": 13774
    },
    "validation": {
      "studies": 296,
      "rows": 2960
    },
    "internal_test": {
      "studies": 296,
      "rows": 2955
    }
  },
  "candidateSearch": {
    "performed": true,
    "candidateCount": 2000,
    "approvedCandidateCount": 142,
    "bestSeed": 2737,
    "bestScore": 0.03476404416506698
  },
  "gateResults": {
    "sourceNotebook57Approved": true,
    "sourceManifestHashVerified": true,
    "allEligibleStudiesAssigned": true,
    "validSplitNames": true,
    "fractionTolerance": true,
    "supportRulesPassed": true,
    "allSeverityClassesInEverySplit": true,
    "noStudyLeakage": true,
    "rowsConserved": true,
    "noDuplicateStudySideLevelRows": true,
    "internalTestSealed": true,
    "officialT

In [4]:
# 4) Gate final y archivos generados
required = [
    "train_manifest.csv", "validation_manifest.csv", "internal_test_manifest.csv",
    "study_split_assignments.csv", "split_distribution.csv", "rare_strata_report.csv",
    "split_coverage.csv", "split_leakage_report.json", "split_summary.json", "split_report.md",
]
missing = [name for name in required if not (OUTPUT_ROOT / name).is_file()]
if missing:
    raise RuntimeError(f"Faltan outputs: {missing}")
if not result["approved"]:
    raise RuntimeError("El split requiere revisión antes del Notebook 59.")
print({
    "status": "APPROVED_FOR_NOTEBOOK_59",
    "outputs": required,
    "internalTestSealed": True,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
    "officialTestAccessed": False,
})

{'status': 'APPROVED_FOR_NOTEBOOK_59', 'outputs': ['train_manifest.csv', 'validation_manifest.csv', 'internal_test_manifest.csv', 'study_split_assignments.csv', 'split_distribution.csv', 'rare_strata_report.csv', 'split_coverage.csv', 'split_leakage_report.json', 'split_summary.json', 'split_report.md'], 'internalTestSealed': True, 'humanReviewRequired': True, 'notClinicalDiagnosis': True, 'officialTestAccessed': False}
